# Notebook 03 — Entrenamiento de Modelos

Entrenamiento y comparación de clasificadores supervisados para detección de fraude.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, recall_score

from src.train import train
from src.config import PROCESSED_DATA_DIR, SUPPORTED_MODELS

## 1. Cargar dataset procesado

In [ ]:
dataset_path = PROCESSED_DATA_DIR / 'messages.csv'

if dataset_path.exists():
    df = pd.read_csv(dataset_path)
    print(f'Dataset cargado: {len(df)} mensajes')
    print(df['label'].value_counts())
else:
    # Dataset de demostración si no hay datos reales
    print(f'No se encontró {dataset_path}. Usando dataset de demostración.')
    print('Ejecuta primero: python main.py prepare')
    df = pd.DataFrame({
        'message': [
            'Your account is blocked verify at http://fake.com now',
            'Hello how are you doing today friend',
            'URGENT send your password and PIN immediately',
            'Reminder your appointment is tomorrow at 10am',
            'You won $10000 send card number to claim prize',
            'Thank you for your order it will arrive Friday',
            'Click here to verify your blocked account http://bank-fake.com',
            'Your package could not be delivered reschedule now',
        ] * 50,
        'label': ['fraudulent', 'legitimate', 'fraudulent', 'legitimate',
                  'fraudulent', 'legitimate', 'fraudulent', 'fraudulent'] * 50
    })
    print(f'Dataset de demostración: {len(df)} mensajes')

## 2. Entrenar y comparar todos los modelos

In [ ]:
results = {}

for model_name in SUPPORTED_MODELS:
    print(f'\nEntrenando {model_name}...')
    try:
        result = train(df, model_name=model_name, save=(model_name == 'logistic_regression'))
        y_pred = result['model'].predict(result['X_test'])
        y_test = result['y_test']
        int_to_label = result['int_to_label']

        # Encontrar índice de 'fraudulent'
        fraud_idx = next((k for k, v in int_to_label.items() if v == 'fraudulent'), None)

        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'f1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
            'recall_fraudulent': recall_score(
                y_test == fraud_idx, y_pred == fraud_idx, zero_division=0
            ) if fraud_idx is not None else None,
        }
        results[model_name] = metrics
        print(f'  Accuracy: {metrics["accuracy"]:.4f} | F1: {metrics["f1_weighted"]:.4f} | Recall fraude: {metrics["recall_fraudulent"]}')
    except Exception as e:
        print(f'  Error: {e}')
        results[model_name] = None

## 3. Tabla comparativa

In [ ]:
import matplotlib.pyplot as plt

valid_results = {k: v for k, v in results.items() if v is not None}
if valid_results:
    comparison_df = pd.DataFrame(valid_results).T.round(4)
    print('\nComparación de modelos:')
    print(comparison_df)

    ax = comparison_df[['accuracy', 'f1_weighted']].plot(
        kind='bar', rot=30, figsize=(10, 4)
    )
    plt.title('Comparación de modelos — Accuracy y F1')
    plt.ylabel('Métrica')
    plt.tight_layout()
    plt.show()
else:
    print('No hay resultados para mostrar.')

## 4. Impacto de las características manuales

Compara el mejor modelo con y sin características estructurales.

In [ ]:
for use_manual in [True, False]:
    result = train(df, model_name='logistic_regression', use_manual_features=use_manual, save=False)
    y_pred = result['model'].predict(result['X_test'])
    acc = accuracy_score(result['y_test'], y_pred)
    f1 = f1_score(result['y_test'], y_pred, average='weighted', zero_division=0)
    label = 'Con características manuales' if use_manual else 'Solo TF-IDF'
    print(f'{label}: Accuracy={acc:.4f} | F1={f1:.4f} | Features={result["X_train"].shape[1]}')